Reinforcement Learning

In [1]:
import numpy as np

# Simple environment: 1D grid, 5 states (0-4), goal is state 4
n_states = 5
n_actions = 2  # 0=left, 1=right

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

def get_reward(state):
    return 10 if state == 4 else -1

def step(state, action):
    if action == 1:  # right
        next_state = min(state+1, n_states-1)
    else:  # left
        next_state = max(state-1, 0)
    reward = get_reward(next_state)
    done = (next_state == 4)
    return next_state, reward, done

for episode in range(episodes):
    state = 0
    done = False
    while not done:
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        next_state, reward, done = step(state, action)

        # Q-Learning update rule
        best_next_q = np.max(Q[next_state])
        Q[state, action] += alpha * (reward + gamma*best_next_q - Q[state, action])

        state = next_state

print("Learned Q-table:\n", Q.round(2))

# Test learned policy
state = 0
path = [state]
while state != 4:
    action = np.argmax(Q[state])
    state, _, _ = step(state, action)
    path.append(state)
print("Learned optimal path:", path)

Learned Q-table:
 [[ 3.12  4.58]
 [ 3.1   6.2 ]
 [ 4.57  8.  ]
 [ 6.18 10.  ]
 [ 0.    0.  ]]
Learned optimal path: [0, 1, 2, 3, 4]


In [2]:
import random
import numpy as np
from collections import deque
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Experience Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (np.array(state), np.array(action), np.array(reward, dtype=np.float32),
                np.array(next_state), np.array(done, dtype=np.float32))

    def __len__(self):
        return len(self.buffer)

# 2. Build Q-Network Architecture
def build_q_network(state_shape, num_actions):
    model = models.Sequential([
        layers.Input(shape=state_shape),
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(num_actions, activation="linear") # Outputs Q-value for each action
    ])
    return model

# 3. Deep Q-Network Agent
class DQNAgent:
    def __init__(self, state_shape, num_actions):
        self.state_shape = state_shape
        self.num_actions = num_actions

        # Hyperparameters
        self.gamma = 0.99           # Discount factor
        self.epsilon = 1.0          # Exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 32

        # Networks & Memory
        self.memory = ReplayBuffer()
        self.model = build_q_network(state_shape, num_actions)
        self.target_model = build_q_network(state_shape, num_actions)
        self.target_model.set_weights(self.model.get_weights())
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def select_action(self, state):
        # Epsilon-greedy exploration strategy
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.num_actions)
        q_values = self.model(np.expand_dims(state, axis=0), training=False)
        return np.argmax(q_values[0])

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return

        # Sample mini-batch from experience replay
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # Compute Target Q-values: R + gamma * max(Q_target(s', a'))
        next_q_values = self.target_model(next_states, training=False)
        max_next_q = np.max(next_q_values, axis=1)
        target_q_values = rewards + (1 - dones) * self.gamma * max_next_q

        with tf.GradientTape() as tape:
            # Current predicted Q-values
            all_q_values = self.model(states, training=True)
            # Mask to gather Q-values corresponding to executed actions
            masks = tf.one_hot(actions, self.num_actions)
            predicted_q_values = tf.reduce_sum(all_q_values * masks, axis=1)

            # Loss = Mean Squared Error (MSE) between target Q and predicted Q
            loss = tf.keras.losses.MSE(target_q_values, predicted_q_values)

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

        # Decay exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target_network(self):
        self.target_model.set_weights(self.model.get_weights())

# 4. Simulation Execution Loop
state_shape = (4,)   # Example: CartPole state space shape
num_actions = 2      # Example: CartPole discrete actions [0: left, 1: right]

agent = DQNAgent(state_shape, num_actions)

# Synthetic Step Simulation
state = np.random.rand(*state_shape)
for step in range(50):
    action = agent.select_action(state)
    next_state = np.random.rand(*state_shape)
    reward = 1.0
    done = False if step < 49 else True

    # Store experience step in replay buffer
    agent.memory.push(state, action, reward, next_state, done)

    # Train Agent
    agent.train_step()
    state = next_state

    # Periodically synchronize target network
    if step % 10 == 0:
        agent.update_target_network()

print("DQN Training step complete! Current exploration rate (epsilon):", round(agent.epsilon, 4))

DQN Training step complete! Current exploration rate (epsilon): 0.9092
